In [2]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns


import os, sys, shutil, importlib, glob
from tqdm.notebook import tqdm

#import libraries for motif analysis

from celloracle import motif_analysis as ma
import celloracle as co
co.__version__

#Figure configurations
%config InlineBackend.figure_format = 'retina'

plt.rcParams['figure.figsize'] = [6, 4.5]
plt.rcParams["savefig.dpi"] = 300


/opt/conda/envs/celloracle2/lib/python3.8/site-packages/logomaker/src/validate.py:98: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if matrix_type is 'information':
/opt/conda/envs/celloracle2/lib/python3.8/site-packages/logomaker/src/validate.py:104: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif matrix_type is 'probability':


In [3]:
#Set working directory
os.chdir("/storage1/fs1/jmillman/Active/DigitalTwin")

# TSS Annotation

In [4]:
#Load scATAC-seq peak list.
all_peaks = pd.read_csv("checkpoints/cicero/all_peaks.csv", index_col=0)

all_peaks  = all_peaks.x.values

print(all_peaks)

#Its important to have and "_" instead of a "-",
#If not, modify the peaks.csv file


['chr1_9702_10718' 'chr1_28874_29754' 'chr1_180651_181730' ...
 'chrY_56869526_56870506' 'chrY_56870686_56871821'
 'chrY_56873337_56874412']


In [5]:
# Load Cicero coaccessibility scores.
cicero_connections = pd.read_csv("checkpoints/cicero/conns.csv",
                                       index_col=0)


In [6]:
#Annotate transcription start sites (TSSs)

## Please make sure to specify the correct reference genome here
tss_annotated = ma.get_tss_info(peak_str_list=all_peaks, ref_genome='hg38')


que bed peaks: 569068
tss peaks in que: 30368


In [7]:
# Check results
print(tss_annotated.tail())


         chr      start        end gene_short_name strand
30363  chr11  126654906  126656247    LOC101929427      +
30364   chr5  149549250  149550272         CSNK1A1      -
30365   chr5  149551116  149552064         CSNK1A1      -
30366  chr20   10673733   10675315            JAG1      -
30367   chr9  122228188  122229268            LHX6      -


In [8]:
#Integrate TSS info and cicero connections

integ_tss_cicero = ma.integrate_tss_peak_with_cicero(tss_peak=tss_annotated, 
                                           cicero_connections=cicero_connections)



print(integ_tss_cicero.shape)

(1958175, 3)


In [9]:
#Filter peaks

proc_peak = integ_tss_cicero[integ_tss_cicero.coaccess >= 0.8]
proc_peak = proc_peak[["peak_id", "gene_short_name"]].reset_index(drop=True)


print(proc_peak.shape)


(27944, 2)


In [10]:
#Save data
proc_peak.to_csv("checkpoints/proc_peak.csv")


# Motif Scan

In [11]:
import celloracle as co
from celloracle import motif_analysis as ma
from celloracle.utility import save_as_pickled_object
co.__version__

'0.18.0'

In [17]:
os.chdir("/storage1/fs1/jmillman/Active/DigitalTwin")
ref_genome = "hg38"
genome_installation = ma.is_genome_installed(ref_genome=ref_genome)
print(ref_genome, "installation: ", genome_installation)


hg38 installation:  True


In [ ]:
'''
#If the genome is already installed, skip this chunk
import genomepy
genomepy.install_genome(name="hg38", provider="UCSC", genomes_dir="Final_Code_Data")

os.chdir("/storage1/fs1/jmillman/Active/RotationStudents/Oracle/Final_Code_Data")
ref_genome = "hg38"
genome_installation = ma.is_genome_installed(ref_genome=ref_genome)
print(ref_genome, "installation: ", genome_installation)
'''


In [19]:
# Load annotated peak data.
proc_peak = pd.read_csv("checkpoints/proc_peak.csv",
                              index_col=0)
print(proc_peak.head(2))


                     peak_id gene_short_name
0  chr10_100009426_100010396           DNMBP
1  chr10_100081035_100082092            CPN1


In [20]:
# Check data

def decompose_chrstr(peak_str):
    """
    Args:
        peak_str (str): peak_str. e.g. 'chr1_3094484_3095479'
        
    Returns:
        tuple: chromosome name, start position, end position
    """
    
    *chr_, start, end = peak_str.split("_")
    chr_ = "_".join(chr_)
    return chr_, start, end

from genomepy import Genome

def check_peak_format(peaks_df, ref_genome):
    """
    Check peak format. 
     (1) Check chromosome name. 
     (2) Check peak size (length) and remove sort DNA sequences (<5bp)
    
    """
    
    df = peaks_df.copy()
    
    n_peaks_before = df.shape[0]
    
    # Decompose peaks and make df
    decomposed = [decompose_chrstr(peak_str) for peak_str in df["peak_id"]]
    df_decomposed = pd.DataFrame(np.array(decomposed), index=peaks_df.index)
    df_decomposed.columns = ["chr", "start", "end"]
    df_decomposed["start"] = df_decomposed["start"].astype(int)
    df_decomposed["end"] = df_decomposed["end"].astype(int)
    
    # Load genome data
    genome_data = Genome(ref_genome)
    all_chr_list = list(genome_data.keys())
    
    
    # DNA length check
    lengths = np.abs(df_decomposed["end"] - df_decomposed["start"])
    
    
    # Filter peaks with invalid chromosome name
    n_threshold = 5
    df = df[(lengths >= n_threshold) & df_decomposed.chr.isin(all_chr_list)]
    
    # DNA length check
    lengths = np.abs(df_decomposed["end"] - df_decomposed["start"])
    
    # Data counting
    n_invalid_length = len(lengths[lengths < n_threshold])
    n_peaks_invalid_chr = n_peaks_before - df_decomposed.chr.isin(all_chr_list).sum()
    n_peaks_after = df.shape[0]
    
    
    #
    print("Peaks before filtering: ", n_peaks_before)
    print("Peaks with invalid chr_name: ", n_peaks_invalid_chr)
    print("Peaks with invalid length: ", n_invalid_length)
    print("Peaks after filtering: ", n_peaks_after)
    
    return df


In [21]:
proc_peak = check_peak_format(proc_peak, ref_genome)

Peaks before filtering:  27944
Peaks with invalid chr_name:  0
Peaks with invalid length:  0
Peaks after filtering:  27944


In [22]:
# Instantiate TFinfo object
tfi = ma.TFinfo(peak_data_frame=proc_peak, 
                ref_genome=ref_genome) 


In [23]:
%%time
# Scan motifs. !!CAUTION!! This step may take several hours if you have many peaks!
tfi.scan(fpr=0.02, 
         motifs=None,  # "None" = default motifs are loaded.
         verbose=True)
#Usually takes from 20 to 60 min

No motif data entered. Loading default motifs for your species ...
 Default motif for vertebrate: gimme.vertebrate.v5.0. 
 For more information, please see https://gimmemotifs.readthedocs.io/en/master/overview.html 

Initiating scanner... 



2025-10-23 00:18:36,021 - DEBUG - using background: genome hg38 with size 200


Calculating FPR-based threshold. This step may take substantial time when you load a new ref-genome. It will be done quicker on the second time. 



2025-10-23 00:18:52,050 - DEBUG - determining FPR-based threshold


Motif scan started .. It may take long time.



Scanning:   0%|          | 0/25626 [00:00<?, ? sequences/s]

CPU times: user 25min 54s, sys: 30.5 s, total: 26min 25s
Wall time: 26min 49s


In [24]:
# Save tfinfo object

tfi.to_hdf5(file_path="checkpoints/DT_27944_peaks_tfi.celloracle.tfinfo")


No motif data entered. Loading default motifs for your species ...
 Default motif for vertebrate: gimme.vertebrate.v5.0. 
 For more information, please see https://gimmemotifs.readthedocs.io/en/master/overview.html 

Initiating scanner... 



2025-10-23 00:45:25,665 - DEBUG - using background: genome hg38 with size 200


Calculating FPR-based threshold. This step may take substantial time when you load a new ref-genome. It will be done quicker on the second time. 

Motif scan started .. It may take long time.



Scanning:   0%|          | 0/25626 [00:00<?, ? sequences/s]

In [25]:
# Check motif scan results
print(tfi.scanned_df.head(2))

                     seqname           motif_id factors_direct  \
0  chr10_100009426_100010396  GM.5.0.Mixed.0001                  
1  chr10_100009426_100010396  GM.5.0.Mixed.0001                  

  factors_indirect     score  pos  strand  
0        EGR1, SRF  8.578792  370      -1  
1        EGR1, SRF  8.319869  629      -1  


In [26]:
# Reset filtering 
tfi.reset_filtering()

# Do filtering
tfi.filter_motifs_by_score(threshold=10)

# Format post-filtering results.
tfi.make_TFinfo_dataframe_and_dictionary(verbose=True)


Filtering finished: 15265744 -> 3049130
1. Converting scanned results into one-hot encoded dataframe.


  0%|          | 0/25626 [00:00<?, ?it/s]

2. Converting results into dictionaries.


  0%|          | 0/21832 [00:00<?, ?it/s]

  0%|          | 0/1094 [00:00<?, ?it/s]

In [29]:
df = tfi.to_dataframe()

In [30]:
# Save result as a dataframe

df.to_parquet("checkpoints/DT_27944_peaks_base_GRN_df.parquet")

In [ ]:
# Choques de estrella, piel con piel. - Dinamita by Rafa Arreguin